In [1]:
import os
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# This works when Jupyter is started from either the repo root or UniDepthLSS/.
REPO = Path.cwd().resolve()
if REPO.name == "UniDepthLSS":
    REPO = REPO.parent
if not (REPO / "UniDepthLSS").is_dir():
    raise RuntimeError("Start Jupyter from the UniDepth_BEV repository.")

default_ckpt = (REPO / "27_May-VGGTBeV/runs"
                / "unidepth_lss_img448x798_bev128/checkpoints/best_model.pt")
NUSCENES_ROOT = Path(os.getenv("NUSCENES_ROOT", "/data/adeel/data/nuscenes"))
UNIDEPTH_ROOT = Path(os.getenv("UNIDEPTH_ROOT", str(REPO.parent / "UniDepth")))
CKPT_PATH = Path(os.getenv("UNIDEPTHLSS_CHECKPOINT", str(default_ckpt)))

sys.path[:0] = [str(REPO / "UniDepthLSS"), str(UNIDEPTH_ROOT)]

from dataset_nuscenes import NuScenesBEVDataset
from model import UniDepthLSS
from train_utils import BinaryIoU

IMG_SIZE = (448, 798)
BEV_SIZE = 128
BEV_RES = 0.5
FEATURE_DIM = 128
MIN_VIS = 2
IOU_THRESHOLD = 0.5
BATCH_SIZE = 1
NUM_WORKERS = 4
GPU = 1

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")
if GPU >= torch.cuda.device_count():
    raise RuntimeError(f"GPU {GPU} is not visible.")
device = torch.device(f"cuda:{GPU}")
torch.cuda.set_device(device)

for path, label in (
    (NUSCENES_ROOT, "nuScenes"),
    (UNIDEPTH_ROOT, "UniDepth checkout"),
    (CKPT_PATH, "checkpoint"),
):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")

print(f"Checkpoint: {CKPT_PATH}")
print(f"Device: {device} ({torch.cuda.get_device_name(GPU)})")

Checkpoint: /home/adeel/UniDepth_BEV/27_May-VGGTBeV/runs/unidepth_lss_img448x798_bev128/checkpoints/best_model.pt
Device: cuda:1 (NVIDIA RTX 6000 Ada Generation)


In [2]:
val_set = NuScenesBEVDataset(
    dataroot=str(NUSCENES_ROOT),
    version="v1.0-trainval",
    split="val",
    img_size=IMG_SIZE,
    bev_size=BEV_SIZE,
    bev_res=BEV_RES,
    augment=False, return_visibility=True,
)
val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
saved_size = tuple(ckpt.get("config", {}).get("image_size", ()))
if saved_size != IMG_SIZE:
    raise ValueError(f"Checkpoint image size is {saved_size}, expected {IMG_SIZE}.")

model = UniDepthLSS(
    img_height=IMG_SIZE[0],
    img_width=IMG_SIZE[1],
    num_classes=1,
    feature_channels=FEATURE_DIM,
)
missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
missing = [name for name in missing if not name.startswith("projector.backbone.")]
if missing or unexpected:
    raise RuntimeError(f"Checkpoint mismatch: missing={missing}, unexpected={unexpected}")
model = model.to(device).eval()

print(f"Validation samples: {len(val_set):,}")
print(f"Loaded epoch {ckpt['epoch']} (saved IoU: {ckpt['val_iou']:.4f})")

/home/adeel/anaconda3/envs/VGGT/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/adeel/UniDepth/unidepth/utils/chamfer_distance.py:9: UserWarning: !! To run evaluation you need KNN. Please compile KNN: `cd unidepth/ops/knn with && bash compile.sh`.
  warnings.warn(
xFormers not available
xFormers not available


Cannot import NystromAttention, you can not run original UniDepth. UniDepthV2 is available.
Not loading pretrained weights for backbone
Validation samples: 6,019
Loaded epoch 10 (saved IoU: 0.4940)


In [3]:
iou_all = BinaryIoU(threshold=IOU_THRESHOLD)
iou_visible = BinaryIoU(threshold=IOU_THRESHOLD)

with torch.inference_mode():
    for images, intrinsics, extrinsics, target, visibility in tqdm(
        val_loader, desc="Validation"
    ):
        images = images.to(device, non_blocking=True)
        intrinsics = intrinsics.to(device, non_blocking=True)
        extrinsics = extrinsics.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        visibility = visibility.to(device, non_blocking=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(images, intrinsics, extrinsics)

        iou_all.update(logits, target)
        iou_visible.update(
            logits, target, valid_mask=visibility >= MIN_VIS
        )

print(f"\nVehicle IoU (no visibility mask): {iou_all.compute():.4f}")
print(f"Vehicle IoU (visibility >= {MIN_VIS}): {iou_visible.compute():.4f}")

Validation:   0%|          | 0/6019 [00:00<?, ?it/s]


Vehicle IoU (no visibility mask): 0.4940
Vehicle IoU (visibility >= 2): 0.5285
